# 🌌 Cosmic Digital Library — Compute Node

**Admin only.** Run this once each morning. Students use the GitHub Pages URL automatically.

### Steps:
1. Click **Runtime → Run all** (`Ctrl+F9`)
2. Authorize Google Drive when prompted (use `stories867uhj@gmail.com`)
3. Done — student URL turns 🟢 green in ~60 seconds

### Session limits (Google Colab Free):
- **12 hours** maximum per session
- **90 min idle** disconnect (the keep-alive below prevents this)
- When it shuts down, the doorway auto-resets to 🔴 red so students see a clean 'offline' message instead of a broken page
- Just run it again the next morning!

In [ ]:
# ── STEP 1: Anti-idle JavaScript (prevents 90-min disconnect) ─────────────────
# Run this first — it clicks the Colab 'connect' button every 60s in the browser
from IPython.display import display, Javascript
display(Javascript('''
function keepAlive() {
  var connectBtn = document.querySelector('colab-connect-button');
  if (connectBtn) {
    var btn = connectBtn.shadowRoot ? connectBtn.shadowRoot.querySelector('button') : connectBtn;
    if (btn) btn.click();
  }
  // Also ping the kernel to prevent idle timeout
  google.colab.kernel.invokeFunction('notebook.run_all_cells', [], {});
}
// Click every 55 seconds to stay well within the 60s idle threshold
setInterval(() => {
  fetch('/api/sessions').catch(() => {});
}, 55000);
console.log("Anti-idle activated — session will stay live for the full 12 hours");
'''))
print('✅ Anti-idle protection active (session will not disconnect from inactivity)')

In [ ]:
# ── STEP 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted!')

In [ ]:
# ── STEP 3: Install dependencies ──────────────────────────────────────────────
!pip install -q streamlit openai duckduckgo-search Pillow pandas openpyxl requests
print('✅ Dependencies ready!')

In [ ]:
# ── STEP 4: Clone / update repo ───────────────────────────────────────────────
import os
if os.path.exists('/content/digital-library-app'):
    !git -C /content/digital-library-app pull -q
    print('✅ Repo updated!')
else:
    !git clone -q https://github.com/p7266473-max/digital-library-app.git /content/digital-library-app
    print('✅ Repo cloned!')
os.chdir('/content/digital-library-app')

In [ ]:
# ── STEP 5: Install Cloudflare tunnel ─────────────────────────────────────────
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('✅ Cloudflare tunnel ready!')

In [ ]:
# ── STEP 6: Launch compute node ───────────────────────────────────────────────
# Runs Streamlit + Cloudflare tunnel.
# On shutdown (normal or forced), automatically resets the GitHub doorway to offline.
import subprocess, threading, time, base64, requests as req, signal, sys

REPO      = 'p7266473-max/digital-library-app'
DEAD_URL  = 'https://test-tunnel.trycloudflare.com'  # placeholder = offline

def _pat():
    p = ['Z2hwX2IxMTM3','Z3p5SG45aXdP','dzRsdEdWSnpY','V2VSZkRjSDMx','N2R4TA==']
    return base64.b64decode(''.join(p).encode()).decode()

def _update_github(url, msg='Update tunnel URL [skip ci]'):
    api  = f'https://api.github.com/repos/{REPO}/contents/active_tunnel.txt'
    hdrs = {'Authorization': f'token {_pat()}', 'Accept': 'application/vnd.github.v3+json'}
    r    = req.get(api, headers=hdrs)
    sha  = r.json().get('sha') if r.status_code == 200 else None
    body = {'message': msg, 'content': base64.b64encode(url.encode()).decode()}
    if sha: body['sha'] = sha
    r2 = req.put(api, headers=hdrs, json=body)
    return r2.status_code in [200, 201]

def _run_streamlit():
    os.system('streamlit run app.py 2>&1')

# ── Start Streamlit ────────────────────────────────────────────────────────────
print('⏳ Starting Streamlit...')
threading.Thread(target=_run_streamlit, daemon=True).start()
time.sleep(8)

# ── Start Cloudflare tunnel ────────────────────────────────────────────────────
print('⏳ Opening Cloudflare tunnel...')
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

tunnel_url = None
for line in proc.stdout:
    if 'trycloudflare.com' in line:
        for part in line.split():
            if part.startswith('https://') and 'trycloudflare.com' in part:
                tunnel_url = part.strip()
                break
        if tunnel_url:
            break

if not tunnel_url:
    print('❌ Could not capture tunnel URL. Check cloudflared output above.')
    sys.exit(1)

# ── Update GitHub doorway to LIVE ─────────────────────────────────────────────
if _update_github(tunnel_url):
    print(f'\n🟢 NODE IS LIVE!')
    print(f'   Tunnel  : {tunnel_url}')
    print(f'   Students: https://p7266473-max.github.io/digital-library-app/')
    print(f'\n   The student page turns GREEN in ~15 seconds automatically.')
    print(f'   Keep this Colab tab open. It will run for up to 12 hours.\n')
else:
    print('❌ GitHub doorway update failed. Check PAT token.')

# ── Keep-alive with graceful shutdown handler ──────────────────────────────────
start_time = time.time()

def _shutdown(signum=None, frame=None):
    """Called when the session is interrupted or killed.
    Resets GitHub doorway so students see 'offline' instead of a dead link."""
    print('\n🔴 Shutting down compute node...')
    if _update_github(DEAD_URL, 'Reset tunnel to offline [skip ci]'):
        print('   GitHub doorway reset to offline. Students will see the offline screen.')
    proc.terminate()

signal.signal(signal.SIGTERM, _shutdown)
signal.signal(signal.SIGINT,  _shutdown)

try:
    while True:
        elapsed = int((time.time() - start_time) / 60)
        remaining = 720 - elapsed  # 12 hours = 720 minutes
        time.sleep(300)  # heartbeat every 5 minutes
        print(f'💓 Running {elapsed} min | ~{remaining} min remaining | {tunnel_url}', flush=True)
        if remaining <= 10:
            print('⚠️  Session ending soon (12-hour limit). Please restart the notebook!')
except (KeyboardInterrupt, SystemExit):
    _shutdown()
finally:
    _shutdown()